In [10]:
# rebuild pathway layer in multiplex

In [11]:
#!/usr/bin/env python3
"""
rebuild_pathway_layer.py
=========================
Builds a new pathway_layer.tsv for your MultiXrank multiplex.

Logic:
  1. Load all current multiplex cell-type layer gene symbols
  2. Load Reactome PPIs  → filter to multiplex genes → gene symbol pairs
  3. Load KEGG PPIs      → filter to multiplex genes → gene symbol pairs
  4. Merge, deduplicate, write pathway_layer.tsv (tab-separated, no header)

Output format (matches what MultiXrank expects):
  GENE_A\tGENE_B
  GENE_A\tGENE_C
  ...
"""
from pathlib import Path
from collections import defaultdict
import pandas as pd
import re

In [12]:
# ── File paths ────────────────────────────────────────────────────────────────
MULTIPLEX_DIR         = Path("../multixrank_input_consolidated/multiplex/gene_multiplex")
REACTOME_INTERACTIONS = Path("reactome_interactions_human.txt")
KEGG_PATHWAYS_CSV     = Path("kegg_pathways_with_names_symbols.csv")
UNIPROT_SYM_CSV       = Path("../../og_gene_set/ppi_chloe/d_uniprot_to_symbol_20251120.csv")
OUTPUT_TSV            = MULTIPLEX_DIR / "pathway_layer.tsv"


In [13]:
# ── Step 1 — Load UniProt → gene symbol map ───────────────────────────────────
print("="*70)
print("STEP 1 — Load UniProt → gene symbol map")
print("="*70)

sym_df = pd.read_csv(UNIPROT_SYM_CSV)
sym_df.columns = sym_df.columns.str.strip().str.lower()
uid_col = next(
    c for c in sym_df.columns
    if c in ("uniprotid", "uniprot_id", "uniprot")
)
sym_col = next(
    c for c in sym_df.columns
    if c in ("symbol", "gene_symbol", "genesymbol")
)
uniprot_to_symbol: dict[str, str] = {}
for _, row in sym_df.iterrows():
    uid = str(row[uid_col]).strip().upper()
    sym = str(row[sym_col]).strip().upper()
    if uid and sym and uid != "NAN" and sym != "NAN":
        uniprot_to_symbol[uid] = sym

print(f"  Loaded {len(uniprot_to_symbol):,} UniProt → symbol entries")

STEP 1 — Load UniProt → gene symbol map
  Loaded 165,580 UniProt → symbol entries


In [14]:
# ── Step 2 — Load multiplex gene symbols ─────────────────────────────────────
print("\n" + "="*70)
print("STEP 2 — Load multiplex gene symbols from all cell-type layer TSVs")
print("="*70)

multiplex_symbols: set[str] = set()
layer_files = sorted(MULTIPLEX_DIR.glob("*_layer.tsv"))

# Only cell-type layers, not pathway/LR layers themselves
cell_type_layers = [
    lf for lf in layer_files
    if not any(x in lf.name.lower() for x in
               ("pathway", "ligand", "receptor"))
]
print(f"  Cell-type layer files found: {len(cell_type_layers):,}")

for lf in cell_type_layers:
    sub = pd.read_csv(
        lf, sep="\t", header=None,
        names=["g1", "g2"], dtype=str, on_bad_lines="skip"
    )
    for col in ("g1", "g2"):
        multiplex_symbols.update(
            sub[col].dropna().str.strip().str.upper()
        )
multiplex_symbols.discard("NAN")
print(f"  Unique gene symbols across all cell-type layers: {len(multiplex_symbols):,}")

# Sanity check — confirm these look like gene symbols not UniProt IDs
uniprot_pattern = re.compile(
    r"^[OPQ][0-9][A-Z0-9]{3}[0-9]$"
    r"|^[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$"
)
n_look_uniprot = sum(
    1 for s in list(multiplex_symbols)[:200]
    if uniprot_pattern.match(s)
)
print(f"  Sanity: {n_look_uniprot}/200 sampled symbols look like UniProt IDs "
      f"(should be ~0)")
print(f"  Sample symbols: {sorted(multiplex_symbols)[:12]}")


STEP 2 — Load multiplex gene symbols from all cell-type layer TSVs
  Cell-type layer files found: 64
  Unique gene symbols across all cell-type layers: 17,051
  Sanity: 0/200 sampled symbols look like UniProt IDs (should be ~0)
  Sample symbols: ['A1BG', 'A1CF', 'A2M', 'A2ML1', 'A4GALT', 'A4GNT', 'AAAS', 'AACS', 'AADAC', 'AADAT', 'AAGAB', 'AAK1']


In [15]:
# ── Step 3 — Load and filter Reactome interactions ────────────────────────────
print("\n" + "="*70)
print("STEP 3 — Load Reactome PPIs and filter to multiplex genes")
print("="*70)
reactome_edges: set[tuple[str, str]] = set()
if REACTOME_INTERACTIONS.exists():
    df_r = pd.read_csv(REACTOME_INTERACTIONS, sep="\t")
    df_r.columns = df_r.columns.str.strip()
    print(f"  Raw Reactome interactions: {len(df_r):,} rows")
    print(f"  All columns: {list(df_r.columns)}")
    print(f"  First 2 rows:\n{df_r.head(2).to_string()}")

    # ── Find UniProt columns explicitly ──────────────────────────────
    uniprot_cols = [c for c in df_r.columns if "uniprot" in c.lower()]
    print(f"\n  Columns containing 'uniprot': {uniprot_cols}")

    if len(uniprot_cols) >= 2:
        col1 = uniprot_cols[0]
        col2 = uniprot_cols[1]
        print(f"  Using name-based: '{col1}' and '{col2}'")
    else:
        # Fall back to positional — col 0 and col 3
        # Reactome format: col0=interactor1_uniprot, col3=interactor2_uniprot
        col1 = df_r.columns[0]
        col2 = df_r.columns[3]
        print(f"  Using positional fallback: col[0]='{col1}', col[3]='{col2}'")

    # Confirm they are actually different columns
    print(f"\n  Sample col1 values: {df_r[col1].head(3).tolist()}")
    print(f"  Sample col2 values: {df_r[col2].head(3).tolist()}")
    assert col1 != col2, (
        f"ERROR: Both UniProt columns resolved to '{col1}' — "
        f"check file format. All columns: {list(df_r.columns)}"
    )

    uid1 = (
        df_r[col1].astype(str)
        .str.replace("uniprotkb:", "", regex=False)
        .str.strip().str.upper()
    )
    uid2 = (
        df_r[col2].astype(str)
        .str.replace("uniprotkb:", "", regex=False)
        .str.strip().str.upper()
    )

    # Verify not all self-pairs before proceeding
    self_pair_count = (uid1 == uid2).sum()
    print(f"\n  Self-pairs (uid1 == uid2)     : {self_pair_count:,} / {len(df_r):,}")
    if self_pair_count == len(df_r):
        raise ValueError(
            "ALL rows are self-pairs — col1 and col2 are the same data. "
            f"col1='{col1}', col2='{col2}'. "
            f"Check the Reactome file format. Columns: {list(df_r.columns)}"
        )

    # Map to gene symbols
    sym1 = uid1.map(uniprot_to_symbol)
    sym2 = uid2.map(uniprot_to_symbol)

    # Filter: both must map to a symbol AND both must be in multiplex
    both_mapped = sym1.notna() & sym2.notna()
    both_in_mp  = (
        sym1.isin(multiplex_symbols) &
        sym2.isin(multiplex_symbols)
    )
    mask = both_mapped & both_in_mp
    print(f"  Both map to a symbol          : {both_mapped.sum():,}")
    print(f"  Both in multiplex             : {both_in_mp.sum():,}")
    print(f"  Both mapped AND in multiplex  : {mask.sum():,}")

    for s1, s2 in zip(sym1[mask], sym2[mask]):
        if s1 != s2:
            key = (min(s1, s2), max(s1, s2))
            reactome_edges.add(key)
    print(f"  Unique undirected Reactome edges retained: {len(reactome_edges):,}")
else:
    print(f"  ⚠  {REACTOME_INTERACTIONS} not found, skipping Reactome")


STEP 3 — Load Reactome PPIs and filter to multiplex genes
  Raw Reactome interactions: 123,895 rows
  All columns: ['# Interactor 1 uniprot id', 'Interactor 1 Ensembl gene id', 'Interactor 1 Entrez Gene id', 'Interactor 2 uniprot id', 'Interactor 2 Ensembl gene id', 'Interactor 2 Entrez Gene id', 'Interaction type', 'Interaction context', 'Pubmed references']
  First 2 rows:
  # Interactor 1 uniprot id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             Interactor 1 Ensembl gene id Interactor 1 Entrez Gene id Interactor 2

In [16]:
import pickle
import networkx as nx

# ── Step 4 — Load KEGG pathway-gene map + PPI, extract KEGG-annotated edges ──
print("\n" + "="*70)
print("STEP 4 — Extract KEGG-annotated edges from existing PPI")
print("="*70)

kegg_edges: set[tuple[str, str]] = set()

PPI_PKL      = Path("../../og_gene_set/temp/combined26/full_network.pkl")
KEGG_CSV     = Path("kegg_pathways_with_names_symbols.csv")

# ── 4a. Load the PPI ─────────────────────────────────────────────────────────
if not PPI_PKL.exists():
    print(f"  ⚠  PPI not found at {PPI_PKL}, skipping KEGG step")
else:
    print(f"  Loading PPI from {PPI_PKL} …")
    with open(PPI_PKL, "rb") as f:
        ppi_graph = pickle.load(f)

    print(f"  PPI nodes : {ppi_graph.number_of_nodes():,}")
    print(f"  PPI edges : {ppi_graph.number_of_edges():,}")
    print(f"  Sample PPI nodes: {list(ppi_graph.nodes())[:10]}")

    # Normalise PPI node labels to upper-case
    # (check whether they are gene symbols or UniProt IDs first)
    sample_nodes = list(ppi_graph.nodes())[:200]
    n_look_uniprot = sum(1 for n in sample_nodes if uniprot_pattern.match(str(n)))
    print(f"  Sanity: {n_look_uniprot}/200 PPI nodes look like UniProt IDs")

    if n_look_uniprot > 100:
        # PPI uses UniProt IDs → convert to gene symbols
        print("  PPI appears to use UniProt IDs — converting to gene symbols …")
        mapping = {
            n: uniprot_to_symbol.get(str(n).strip().upper())
            for n in ppi_graph.nodes()
        }
        n_mapped   = sum(1 for v in mapping.values() if v is not None)
        n_unmapped = sum(1 for v in mapping.values() if v is None)
        print(f"  Mapped   : {n_mapped:,}")
        print(f"  Unmapped : {n_unmapped:,}")
        # Relabel — drop nodes that didn't map
        ppi_graph = nx.relabel_nodes(
            ppi_graph,
            {k: v for k, v in mapping.items() if v is not None}
        )
        # Remove any nodes that are still UniProt-looking (unmapped)
        to_remove = [
            n for n in ppi_graph.nodes()
            if uniprot_pattern.match(str(n))
        ]
        ppi_graph.remove_nodes_from(to_remove)
        print(f"  PPI after conversion: {ppi_graph.number_of_nodes():,} nodes, "
              f"{ppi_graph.number_of_edges():,} edges")
    else:
        # Already gene symbols — just upper-case them
        print("  PPI appears to use gene symbols — normalising case …")
        ppi_graph = nx.relabel_nodes(
            ppi_graph,
            {n: str(n).strip().upper() for n in ppi_graph.nodes()}
        )

    # ── 4b. Load KEGG pathway → gene set map ─────────────────────────────────
    if not KEGG_CSV.exists():
        print(f"  ⚠  KEGG CSV not found at {KEGG_CSV}, skipping KEGG annotation")
    else:
        df_k = pd.read_csv(KEGG_CSV)
        df_k.columns = df_k.columns.str.strip().str.lower()
        print(f"\n  KEGG CSV shape   : {df_k.shape}")
        print(f"  KEGG CSV columns : {list(df_k.columns)}")

        # Find pathway name column and gene symbol column
        pw_col = next(
            (c for c in df_k.columns
             if "pathway" in c and "name" in c), None
        ) or next(
            (c for c in df_k.columns if "pathway" in c), None
        )
        sym_col_k = next(
            (c for c in df_k.columns
             if c in ("gene_symbol", "gene_name_symbol",
                      )), None
        )

        if not pw_col or not sym_col_k:
            print(f"  ⚠  Cannot find pathway/symbol columns. "
                  f"Found: {list(df_k.columns)}")
        else:
            print(f"  Using columns: pathway='{pw_col}'  symbol='{sym_col_k}'")

            # Build pathway → set of gene symbols
            pathway_to_genes: dict[str, set[str]] = defaultdict(set)
            for _, row in df_k.iterrows():
                pw  = str(row[pw_col]).strip()
                sym = str(row[sym_col_k]).strip().upper()
                if pw and sym and pw != "NAN" and sym != "NAN":
                    pathway_to_genes[pw].add(sym)

            print(f"  Unique KEGG pathways : {len(pathway_to_genes):,}")
            total_genes_kegg = len({g for gs in pathway_to_genes.values() for g in gs})
            print(f"  Unique genes in KEGG : {total_genes_kegg:,}")

            # ── 4c. For each KEGG pathway, keep PPI edges where both
            #        endpoints are in that pathway AND in multiplex ───────────
            print(f"\n  Scanning pathways for PPI edges …")
            pathway_edge_counts: dict[str, int] = {}

            for pw_name, gene_set in pathway_to_genes.items():
                # Restrict to multiplex genes
                pw_mp_genes = gene_set & multiplex_symbols

                if len(pw_mp_genes) < 2:
                    continue

                # Find all PPI edges within this pathway's gene set
                subgraph = ppi_graph.subgraph(pw_mp_genes)
                n_new = 0
                for u, v in subgraph.edges():
                    if u != v:
                        key = (min(u, v), max(u, v))
                        if key not in kegg_edges:
                            kegg_edges.add(key)
                            n_new += 1
                if n_new > 0:
                    pathway_edge_counts[pw_name] = n_new

            print(f"  Unique KEGG-annotated PPI edges : {len(kegg_edges):,}")
            print(f"  Pathways that contributed edges : "
                  f"{len(pathway_edge_counts):,}")
            print(f"\n  Top 15 most-contributing KEGG pathways:")
            for pw, cnt in sorted(
                pathway_edge_counts.items(),
                key=lambda x: x[1], reverse=True
            )[:15]:
                print(f"    {cnt:6,}  {pw}")


STEP 4 — Extract KEGG-annotated edges from existing PPI
  Loading PPI from ../../og_gene_set/temp/combined26/full_network.pkl …
  PPI nodes : 17,053
  PPI edges : 302,564
  Sample PPI nodes: ['ADAM12', 'FHL5', 'KRTAP1-1', 'GUCD1', 'NOTCH2NLA', 'FSTL3', 'KRTAP9-2', 'OTX1', 'KRTAP1-3', 'SPRY4']
  Sanity: 0/200 PPI nodes look like UniProt IDs
  PPI appears to use gene symbols — normalising case …

  KEGG CSV shape   : (39334, 6)
  KEGG CSV columns : ['unnamed: 0', 'gene_name', 'pathway_name', 'gene_name_uniprot', 'gene_name_symbol', 'gene_symbol']
  Using columns: pathway='pathway_name'  symbol='gene_name_symbol'
  Unique KEGG pathways : 370
  Unique genes in KEGG : 7,674

  Scanning pathways for PPI edges …
  Unique KEGG-annotated PPI edges : 22,358
  Pathways that contributed edges : 252

  Top 15 most-contributing KEGG pathways:
     3,465  Ribosome - Homo sapiens (human)
     1,893  Metabolic pathways - Homo sapiens (human)
     1,188  Spliceosome - Homo sapiens (human)
       719  C

In [17]:
# ── Step 5 — Merge and deduplicate ───────────────────────────────────────────
print("\n" + "="*70)
print("STEP 5 — Merge Reactome + KEGG edges")
print("="*70)

all_edges = reactome_edges | kegg_edges

reactome_only = reactome_edges - kegg_edges
kegg_only     = kegg_edges - reactome_edges
both          = reactome_edges & kegg_edges

print(f"  Reactome edges   : {len(reactome_edges):,}")
print(f"  KEGG edges       : {len(kegg_edges):,}")
print(f"  In both          : {len(both):,}")
print(f"  Reactome only    : {len(reactome_only):,}")
print(f"  KEGG only        : {len(kegg_only):,}")
print(f"  Total unique     : {len(all_edges):,}")


STEP 5 — Merge Reactome + KEGG edges
  Reactome edges   : 20,488
  KEGG edges       : 22,358
  In both          : 2,937
  Reactome only    : 17,551
  KEGG only        : 19,421
  Total unique     : 39,909


In [18]:
# ── Step 6 — Final sanity checks ─────────────────────────────────────────────
print("\n" + "="*70)
print("STEP 6 — Sanity checks")
print("="*70)

# All genes in the edge list should be in multiplex
edge_genes = {g for pair in all_edges for g in pair}
not_in_mp  = edge_genes - multiplex_symbols
print(f"  Genes in edge list             : {len(edge_genes):,}")
print(f"  Genes NOT in multiplex (should be 0): {len(not_in_mp):,}")
if not_in_mp:
    print(f"  Sample: {sorted(not_in_mp)[:10]}")

# Degree distribution
degree: dict[str, int] = defaultdict(int)
for a, b in all_edges:
    degree[a] += 1
    degree[b] += 1

deg_series = pd.Series(degree)
print(f"\n  Degree distribution:")
print(f"    Min    : {deg_series.min()}")
print(f"    Median : {deg_series.median():.0f}")
print(f"    Mean   : {deg_series.mean():.1f}")
print(f"    Max    : {deg_series.max()}")
print(f"    Top 10 hubs:")
for gene, deg in deg_series.sort_values(ascending=False).head(10).items():
    print(f"      {gene:12s} : {deg:,}")

# Genes in multiplex but with NO pathway edge
no_edge = multiplex_symbols - edge_genes
print(f"\n  Multiplex genes with no pathway edge: {len(no_edge):,} "
      f"({100*len(no_edge)//max(len(multiplex_symbols),1)}%)")
print(f"  Sample: {sorted(no_edge)[:15]}")


STEP 6 — Sanity checks
  Genes in edge list             : 7,254
  Genes NOT in multiplex (should be 0): 0

  Degree distribution:
    Min    : 1
    Median : 5
    Mean   : 11.0
    Max    : 243
    Top 10 hubs:
      GRB2         : 243
      SRC          : 222
      PIK3R1       : 203
      CTNNB1       : 201
      EP300        : 197
      MYC          : 196
      AKT1         : 192
      HDAC1        : 184
      TRAF6        : 157
      PTPN11       : 143

  Multiplex genes with no pathway edge: 9,797 (57%)
  Sample: ['A1BG', 'A2ML1', 'A4GALT', 'A4GNT', 'AACS', 'AADAC', 'AADAT', 'AAGAB', 'AAK1', 'AAMDC', 'AANAT', 'AAR2', 'AARD', 'AARSD1', 'AASDH']


In [19]:
# ── Step 7 — Write pathway_layer.tsv ─────────────────────────────────────────
print("\n" + "="*70)
print("STEP 7 — Write pathway_layer.tsv")
print("="*70)

# Sort for reproducibility
sorted_edges = sorted(all_edges)

with open(OUTPUT_TSV, "w") as f:
    for a, b in sorted_edges:
        f.write(f"{a}\t{b}\n")

print(f"  Written {len(sorted_edges):,} edges → {OUTPUT_TSV}")
print(f"  File size: {OUTPUT_TSV.stat().st_size / 1024:.1f} KB")

# Quick verification read-back
verify = pd.read_csv(
    OUTPUT_TSV, sep="\t", header=None,
    names=["g1", "g2"], dtype=str
)
print(f"  Verification read-back: {len(verify):,} rows, "
      f"{verify['g1'].nunique() + verify['g2'].nunique()} gene mentions")

print("\n" + "="*70)
print("DONE")
print("="*70)
print(f"  New pathway_layer.tsv : {len(sorted_edges):,} edges")
print(f"  Replaces old file with: 204 edges")
print(f"  Coverage improvement  : {len(sorted_edges)//204}×")
print()
print("Next steps:")
print("  1. Re-run rebuild_reactome_csv.py to regenerate")
print("     reactome_interactions_with_pathways_v2.csv")
print("  2. Re-run the main analysis with --reactome-csv pointing")
print("     to the new CSV")


STEP 7 — Write pathway_layer.tsv
  Written 39,909 edges → ../multixrank_input_consolidated/multiplex/gene_multiplex/pathway_layer.tsv
  File size: 466.6 KB
  Verification read-back: 39,909 rows, 11404 gene mentions

DONE
  New pathway_layer.tsv : 39,909 edges
  Replaces old file with: 204 edges
  Coverage improvement  : 195×

Next steps:
  1. Re-run rebuild_reactome_csv.py to regenerate
     reactome_interactions_with_pathways_v2.csv
  2. Re-run the main analysis with --reactome-csv pointing
     to the new CSV


In [20]:
#!/usr/bin/env python3
"""
verify_pathway_layer.py
========================
Quick sanity checks on the newly built pathway_layer.tsv
"""
from pathlib import Path
from collections import Counter
import pandas as pd
import networkx as nx

# ── Paths — adjust as needed ──────────────────────────────────────────────────
PATHWAY_LAYER  = Path("../multixrank_input_consolidated/multiplex/gene_multiplex/pathway_layer.tsv")
MULTIPLEX_DIR  = Path("../multixrank_input_consolidated/multiplex/gene_multiplex")
REACTOME_CSV   = Path("reactome_interactions_with_pathways_v2.csv")

# ── 1. Load pathway layer ─────────────────────────────────────────────────────
print("=" * 70)
print("1. LOAD PATHWAY LAYER")
print("=" * 70)
df = pd.read_csv(PATHWAY_LAYER, sep="\t", header=None, names=["g1", "g2"], dtype=str)
print(f"  Rows          : {len(df):,}")
print(f"  Sample edges  :")
print(df.head(10).to_string(index=False))

# ── 2. Basic integrity checks ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("2. BASIC INTEGRITY")
print("=" * 70)

# Self-pairs
self_pairs = (df["g1"] == df["g2"]).sum()
print(f"  Self-pairs (should be 0)     : {self_pairs:,}")

# Nulls
nulls = df.isna().sum().sum()
print(f"  Null values (should be 0)    : {nulls:,}")

# Duplicates (both directions)
df["key"] = df.apply(lambda r: tuple(sorted([r["g1"], r["g2"]])), axis=1)
dupes = df.duplicated(subset="key").sum()
print(f"  Duplicate edges (should be 0): {dupes:,}")

# Look like gene symbols not UniProt IDs
import re
uniprot_pattern = re.compile(
    r"^[OPQ][0-9][A-Z0-9]{3}[0-9]$"
    r"|^[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$"
)
all_genes = set(df["g1"]) | set(df["g2"])
n_look_uniprot = sum(1 for g in list(all_genes)[:500] if uniprot_pattern.match(g))
print(f"  Genes looking like UniProt IDs (should be ~0): "
      f"{n_look_uniprot} / {min(500, len(all_genes))} sampled")
print(f"  Sample genes: {sorted(all_genes)[:15]}")

# ── 3. Coverage against multiplex cell-type layers ───────────────────────────
print("\n" + "=" * 70)
print("3. COVERAGE VS MULTIPLEX CELL-TYPE LAYERS")
print("=" * 70)

multiplex_symbols: set[str] = set()
layer_files = sorted(MULTIPLEX_DIR.glob("*_layer.tsv"))
cell_type_layers = [
    lf for lf in layer_files
    if not any(x in lf.name.lower() for x in ("pathway", "ligand", "receptor"))
]
print(f"  Cell-type layer files: {len(cell_type_layers):,}")
for lf in cell_type_layers:
    sub = pd.read_csv(lf, sep="\t", header=None,
                      names=["g1","g2"], dtype=str, on_bad_lines="skip")
    for col in ("g1","g2"):
        multiplex_symbols.update(sub[col].dropna().str.strip().str.upper())
multiplex_symbols.discard("NAN")
print(f"  Total multiplex genes: {len(multiplex_symbols):,}")

pathway_genes = set(df["g1"]) | set(df["g2"])
not_in_mp = pathway_genes - multiplex_symbols
in_mp     = pathway_genes & multiplex_symbols
print(f"  Pathway layer genes           : {len(pathway_genes):,}")
print(f"  In multiplex                  : {len(in_mp):,}")
print(f"  NOT in multiplex (should be 0): {len(not_in_mp):,}")
if not_in_mp:
    print(f"  Sample not-in-multiplex: {sorted(not_in_mp)[:15]}")

mp_with_edge    = multiplex_symbols & pathway_genes
mp_without_edge = multiplex_symbols - pathway_genes
print(f"\n  Multiplex genes WITH pathway edge   : {len(mp_with_edge):,} "
      f"({100*len(mp_with_edge)//len(multiplex_symbols)}%)")
print(f"  Multiplex genes WITHOUT pathway edge: {len(mp_without_edge):,} "
      f"({100*len(mp_without_edge)//len(multiplex_symbols)}%)")
print(f"  Sample without edge: {sorted(mp_without_edge)[:15]}")

# ── 4. Graph properties ───────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("4. GRAPH PROPERTIES")
print("=" * 70)

G = nx.from_pandas_edgelist(df, "g1", "g2")
print(f"  Nodes         : {G.number_of_nodes():,}")
print(f"  Edges         : {G.number_of_edges():,}")
print(f"  Connected components: {nx.number_connected_components(G):,}")

degrees = pd.Series(dict(G.degree()))
print(f"\n  Degree distribution:")
print(f"    Min    : {degrees.min()}")
print(f"    Median : {degrees.median():.0f}")
print(f"    Mean   : {degrees.mean():.1f}")
print(f"    Max    : {degrees.max()}")
print(f"\n  Top 15 hubs:")
for gene, deg in degrees.sort_values(ascending=False).head(15).items():
    print(f"    {gene:15s} : {deg:,}")

# Largest connected component
lcc = max(nx.connected_components(G), key=len)
print(f"\n  Largest connected component: {len(lcc):,} nodes "
      f"({100*len(lcc)//G.number_of_nodes()}% of all nodes)")

# ── 5. Cross-check against reactome CSV ──────────────────────────────────────
print("\n" + "=" * 70)
print("5. CROSS-CHECK VS REACTOME CSV")
print("=" * 70)

if REACTOME_CSV.exists():
    rdf = pd.read_csv(REACTOME_CSV)
    print(f"  Reactome CSV rows    : {len(rdf):,}")
    print(f"  Self-pairs in CSV    : {(rdf['symbol1']==rdf['symbol2']).sum():,}")
    print(f"  Distinct pairs       : {(rdf['symbol1']!=rdf['symbol2']).sum():,}")
    print(f"  With common_pathway  : {rdf['common_pathway'].notna().sum():,}")

    # Build set of symbol pairs from CSV
    csv_pairs = set(
        tuple(sorted([r["symbol1"], r["symbol2"]]))
        for _, r in rdf[
            rdf["symbol1"].notna() &
            rdf["symbol2"].notna() &
            (rdf["symbol1"] != rdf["symbol2"])
        ].iterrows()
    )
    layer_pairs = set(df["key"].tolist())

    in_both          = layer_pairs & csv_pairs
    in_layer_not_csv = layer_pairs - csv_pairs  # KEGG-only edges
    in_csv_not_layer = csv_pairs - layer_pairs  # filtered out (not in multiplex)

    print(f"\n  Pairs in pathway_layer.tsv       : {len(layer_pairs):,}")
    print(f"  Distinct pairs in reactome CSV   : {len(csv_pairs):,}")
    print(f"  In both                          : {len(in_both):,}")
    print(f"  In layer but NOT in CSV (KEGG)   : {len(in_layer_not_csv):,}")
    print(f"  In CSV but NOT in layer          : {len(in_csv_not_layer):,}")
    print(f"  (^ these were filtered: not in multiplex)")

    # Pathway coverage of layer edges
    rdf_filtered = rdf[
        (rdf["symbol1"] != rdf["symbol2"]) &
        rdf["common_pathway"].notna()
    ].copy()
    rdf_filtered["key"] = rdf_filtered.apply(
        lambda r: tuple(sorted([r["symbol1"], r["symbol2"]])), axis=1
    )
    pw_covered = layer_pairs & set(rdf_filtered["key"])
    print(f"\n  Layer edges with named pathway   : {len(pw_covered):,} "
          f"({100*len(pw_covered)//max(len(layer_pairs),1)}%)")
    print(f"  Layer edges without named pathway: {len(layer_pairs)-len(pw_covered):,} "
          f"(KEGG-only, will get fallback label in connectivity analysis)")

    # Top pathways
    print(f"\n  Top 20 pathways in reactome CSV:")
    pathway_counter = Counter(
        p
        for val in rdf_filtered["common_pathway"].dropna()
        for p in str(val).split("|")
        if p.strip() and p.strip().lower() != "pathway"
    )
    for pw, cnt in pathway_counter.most_common(20):
        print(f"    {cnt:6,}  {pw}")
else:
    print(f"  ⚠  {REACTOME_CSV} not found, skipping cross-check")

# ── 6. Summary ────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("6. SUMMARY — WHAT TO EXPECT IN CONNECTIVITY ANALYSIS")
print("=" * 70)
print(f"  Total pathway_layer edges      : {len(df):,}")
print(f"  Edges with named pathway (~50%+): check section 5 above")
print(f"  Genes covered                  : {len(pathway_genes):,} / {len(multiplex_symbols):,} multiplex genes")
print(f"  Self-pairs                     : {self_pairs} (must be 0)")
print(f"  Duplicate edges                : {dupes} (must be 0)")
print(f"  Genes outside multiplex        : {len(not_in_mp)} (must be 0)")
if self_pairs == 0 and dupes == 0 and len(not_in_mp) == 0:
    print(f"\n  ✓ All integrity checks passed")
else:
    print(f"\n  ✗ Issues found — see above")

1. LOAD PATHWAY LAYER
  Rows          : 39,909
  Sample edges  :
  g1      g2
A1CF APOBEC1
 A2M   KLKB1
 A2M    MMP1
 A2M   MMP13
 A2M    MMP3
AAAS    NDC1
AAAS   NUP35
AAMP    EGFR
AAMP    NOD2
AAMP  TBXA2R

2. BASIC INTEGRITY
  Self-pairs (should be 0)     : 0
  Null values (should be 0)    : 0
  Duplicate edges (should be 0): 0
  Genes looking like UniProt IDs (should be ~0): 0 / 500 sampled
  Sample genes: ['A1CF', 'A2M', 'AAAS', 'AAMP', 'AARS1', 'AARS2', 'AASDHPPT', 'AATF', 'ABAT', 'ABCA1', 'ABCA10', 'ABCA2', 'ABCA6', 'ABCA7', 'ABCA8']

3. COVERAGE VS MULTIPLEX CELL-TYPE LAYERS
  Cell-type layer files: 64
  Total multiplex genes: 17,051
  Pathway layer genes           : 7,254
  In multiplex                  : 7,254
  NOT in multiplex (should be 0): 0

  Multiplex genes WITH pathway edge   : 7,254 (42%)
  Multiplex genes WITHOUT pathway edge: 9,797 (57%)
  Sample without edge: ['A1BG', 'A2ML1', 'A4GALT', 'A4GNT', 'AACS', 'AADAC', 'AADAT', 'AAGAB', 'AAK1', 'AAMDC', 'AANAT', 'AAR2', 